In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 0.8 Fitting and Least Squares

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume 0 — Mathematical & Computational Foundations",
    number="0.8",
    title="Fitting and Least Squares",
    blurb="Extracting parameters from data the right way: least squares as a "
    "linear-algebra problem, why the textbook normal equations quietly "
    "lose twice as many digits as the problem demands, nonlinear fitting with honest error bars, "
    "and the ever-present temptation to overfit.",
    difficulty="intermediate",
    estimate="90–120 min",
)

## Notebook overview

Every experiment ends the same way: a cloud of points, a model with a few free
parameters, and the question *what values make the model fit?* The answer, almost
always, is **least squares**. The quiet thesis of this notebook is that least
squares is not a statistics topic bolted onto linear algebra but a linear-algebra
problem wearing a lab coat. Fitting a line, fitting a polynomial, fitting any
model linear in its parameters is exactly the projection of a data vector onto the
column space of a **design matrix**, the same geometry we met in
[§0.4](linear-systems.ipynb) and [§0.5](eigenvalues-svd.ipynb).

That viewpoint yields an immediate dividend. The formula every textbook prints, the
**normal equations** $A^\top A\,\mathbf c = A^\top\mathbf y$, is mathematically
perfect and numerically treacherous: forming $A^\top A$ *squares* the condition
number, so it throws away twice as many digits as the problem itself demands. This
is the conditioning lesson of [§0.4](linear-systems.ipynb) and
[§0.5](eigenvalues-svd.ipynb), now with consequences we can watch on
a real fit. The cure is also from those notebooks: solve through **QR** or the
**SVD**, never through $A^\top A$, which is precisely what `numpy.polyfit` and
`numpy.linalg.lstsq` do under the hood.

From there we build the working toolkit a physicist actually needs: residuals and
the $\chi^2$ statistic, the parameter **covariance matrix** and the error bars it
yields, **nonlinear** fitting when the model is not linear in its parameters
(`scipy.optimize.curve_fit`, an optimisation/root-finding problem that calls back
to [§0.2](root-finding.ipynb) and [§0.7](ode-solvers.ipynb)), **weighted** least
squares for data with unequal uncertainties,
and the cautionary tale that never goes away: **overfitting**, where adding
parameters fits the training points beautifully while tracking the noise and
generalising worse. There are no animations here. A fit and its residuals are
static objects, and a static plot states the lesson more clearly than motion
would.

This notebook is the payoff of the two linear-algebra notebooks before it, and it
feeds everything experimental downstream: extracting rates, exponents, lattice
constants, and spectral peaks is this computation, run again and again.

> **How to read the checks.** Each exercise ends with a `validate` call against an
> independent fact: three solution methods agreeing, a fitted parameter landing
> inside its error bar, a held-out error rising. A ✓ is strong evidence; a ✗ is a
> prompt to *locate the discrepancy*, not a verdict.

> **Scope.** A working review, not a statistics course. The standard references
> are Press et al., *Numerical Recipes* {cite}`numrecipes` (ch. 15), Trefethen &
> Bau, *Numerical Linear Algebra* {cite}`trefethen1997` (least squares and QR),
> and Bevington & Robinson {cite}`bevington` for $\chi^2$ and error analysis.

## Theory in brief

### Least squares as projection

Given data $(x_i, y_i)$ and a model **linear in its parameters** $\mathbf c$, we
assemble the **design matrix** $A$ whose columns are the basis functions evaluated
at the $x_i$ (for a line, the columns are $1$ and $x_i$; for a degree-$d$
polynomial, the powers $x_i^0,\dots,x_i^d$). The fit minimises the sum of squared
residuals,

```{math}
:label: eq-lsq
\min_{\mathbf c}\ \lVert A\mathbf c - \mathbf y\rVert_2^2 .
```

Geometrically, $A\mathbf c$ ranges over the **column space** of $A$, and the
minimiser is the orthogonal **projection** of $\mathbf y$ onto that subspace: the
residual $\mathbf y - A\mathbf c$ is perpendicular to every column of $A$. That
perpendicularity, written out, *is* the next equation.

### The normal equations (correct, but dangerous)

Setting the gradient of {eq}`eq-lsq` to zero gives the **normal equations**

```{math}
:label: eq-normal
A^\top A\,\mathbf c = A^\top\mathbf y .
```

They are exact on paper and the fastest thing to code, which is why every textbook
shows them. They are also the wrong way to compute a fit, for a reason that is
pure [§0.4](linear-systems.ipynb)/[§0.5](eigenvalues-svd.ipynb): forming
$A^\top A$ **squares the condition number**,
$\kappa(A^\top A) = \kappa(A)^2$. Since the achievable accuracy scales like
$\varepsilon\,\kappa$, squaring $\kappa$ doubles the number of digits lost. A
fit that is mildly ill-conditioned in $A$ becomes catastrophically so in
$A^\top A$.

### Solving through QR or the SVD

The fix is to solve {eq}`eq-lsq` *without ever forming* $A^\top A$. Using the QR
factorization $A = QR$ of [§0.4](linear-systems.ipynb) (orthonormal $Q$,
upper-triangular $R$),

```{math}
:label: eq-qr-lsq
A = QR \quad\Longrightarrow\quad R\,\mathbf c = Q^\top\mathbf y ,
```

a single triangular solve whose accuracy depends on $\kappa(A)$, not its square.
The SVD-based **pseudoinverse** $A^+ = V\Sigma^+U^\top$ of
[§0.5](eigenvalues-svd.ipynb) does the same job
and is the most robust of all when $A$ is nearly rank-deficient. These are exactly
what the library routines use: `numpy.linalg.lstsq` is SVD-based, and
`numpy.polyfit` builds on the same idea. Use them; do not hand-roll the normal
equations.

### Goodness of fit, $\chi^2$, and error bars

A fit is not finished until we know how good it is and how uncertain the
parameters are. The **residuals** $r_i = y_i - (A\mathbf c)_i$ give the root-mean-
square error $\mathrm{RMSE} = \sqrt{\tfrac1n\sum r_i^2}$. When the data carry known
uncertainties $\sigma_i$, the right scalar is the chi-square statistic, and the
parameter **covariance** follows from it:

```{math}
:label: eq-chi2
\chi^2 = \sum_i \frac{r_i^2}{\sigma_i^2}, \qquad
\operatorname{cov}(\mathbf c) = \hat\sigma^2\,(A^\top A)^{-1}, \quad
\hat\sigma^2 = \frac{\lVert\mathbf r\rVert^2}{n - p},
```

for $n$ data points and $p$ parameters. The **standard error** of each parameter
is the square root of the corresponding diagonal entry of the covariance. The
covariance formula comes from propagating the data uncertainties through the
linear solve; Press et al., *Numerical Recipes*, §15.4, carry the derivation
out, and Bevington & Robinson {cite}`bevington` develop the statistics. (The
$(A^\top A)^{-1}$ here is a small $p\times p$ object used for uncertainty, not the
route by which we *solve* the fit.)

### Weighted least squares

When points have different uncertainties $\sigma_i$, they should not count
equally. Weighting each residual by $1/\sigma_i^2$ minimises

```{math}
:label: eq-weighted
\min_{\mathbf c}\ \sum_i \frac{\big(y_i - (A\mathbf c)_i\big)^2}{\sigma_i^2}
= \lVert W^{1/2}(A\mathbf c - \mathbf y)\rVert_2^2, \quad W = \operatorname{diag}(1/\sigma_i^2),
```

which is the **maximum-likelihood** fit for independent Gaussian errors (writing
the Gaussian likelihood and taking its logarithm shows the equivalence;
Bevington & Robinson {cite}`bevington` spell it out). Precise
points pull harder; noisy points are allowed to miss.

### Nonlinear least squares

If the model is **nonlinear** in its parameters (a decay rate inside an
exponential, a frequency inside a sine), the residual norm is no longer quadratic
and there is no closed form. We minimise iteratively,

```{math}
:label: eq-nonlin
\min_{\boldsymbol\theta}\ \sum_i \big(y_i - f(x_i;\boldsymbol\theta)\big)^2 ,
```

by Gauss–Newton or Levenberg–Marquardt: a sequence of *linearised* least-squares
steps, each one an instance of everything above, and each one a root-finding/
optimisation step in the spirit of [§0.2](root-finding.ipynb) and
[§0.7](ode-solvers.ipynb) (Press et al., *Numerical Recipes*, §15.5, develop
Levenberg–Marquardt in full). `scipy.optimize.curve_fit`
wraps this and returns the covariance for free.

### Overfitting

Finally, a warning that outlives every method. More parameters *always* reduce the
residual on the data one fits, but past a point they fit the **noise**, not the
signal, and the model generalises worse. The tell is the gap between the error on
the training data and the error on held-out data: the first keeps falling, the
second turns and climbs. This is the first taste of the bias–variance trade that
runs through all of data analysis.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.polynomial import Polynomial
from scipy.linalg import qr, solve_triangular
from scipy.optimize import curve_fit

from ecp import validate

np.set_printoptions(precision=4, suppress=True)

## Exercise 1 — Linear least squares three ways

A line is the simplest fit, and computing it three different
ways (the normal equations {eq}`eq-normal`, a QR solve {eq}`eq-qr-lsq`, and the
library `lstsq`) is the cleanest way to see that on a well-conditioned problem
they all agree. The disagreements come later, on the hard problems.

Take the explicit dataset $x = $ `linspace(0, 10, 50)` and $y = 2 + 3x +
\varepsilon$ with $\varepsilon = $ `default_rng(0).normal(0, 1, 50)`, and fit the
line $y = a + bx$, whose design matrix has columns $1$ and $x$ ({eq}`eq-lsq`,
built with `numpy.vander`).

1. Solve via the normal equations (`numpy.linalg.solve` on $A^\top A$).
2. Solve via QR (`scipy.linalg.qr` + `scipy.linalg.solve_triangular`).
3. Solve via `numpy.linalg.lstsq`, and confirm all three coefficient vectors
   agree. {numref}`fig-fit-linear` shows the data and the line.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    c_normal,
    c_qr,
    "the normal-equation and QR solutions agree on the well-conditioned fit",
    rtol=1e-8,
)
validate.close(
    c_normal,
    c_lstsq,
    "the normal-equation and lstsq solutions agree on the well-conditioned fit",
    rtol=1e-8,
)

## Exercise 2 — The design matrix and the projection picture

Least squares *is* a projection ({eq}`eq-lsq`), and this
exercise makes the geometry literal. For a polynomial fit the design matrix is the
**Vandermonde** matrix, columns $x^0, x^1, \dots, x^d$; the fitted values
$A\mathbf c$ are the projection of $\mathbf y$ onto the column space of $A$. We can
compute that projection two independent ways and check they coincide: the fit
$A\mathbf c$ from `lstsq`, and the explicit orthogonal projector $Q Q^\top\mathbf
y$ built from the QR factors.

For the explicit dataset $x = $ `linspace(-2, 2, 40)`,
$y = 1 + 0.5x - 0.3x^2 + 0.2x^3 + \varepsilon$ with $\varepsilon = $
`default_rng(2).normal(0, 0.4, 40)`:

1. Build the degree-3 Vandermonde design matrix (`numpy.vander`) and fit it with
   `numpy.linalg.lstsq`.
2. Confirm the vector of fitted values equals the projection of $\mathbf y$ onto
   $\operatorname{col}(A)$, computed independently as $Q Q^\top\mathbf y$ from
   the `scipy.linalg.qr` factors.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    fit_values,
    projection,
    "the least-squares fit is the orthogonal projection of y onto the column space of A",
    rtol=1e-8,
)

## Exercise 3 — Why not the normal equations: squaring the condition number

This is the centrepiece, and it is the
conditioning lesson of [§0.4](linear-systems.ipynb) and
[§0.5](eigenvalues-svd.ipynb) applied directly to fitting. Forming $A^\top A$
squares the condition number, $\kappa(A^\top A) = \kappa(A)^2$, so the normal
equations operate on a matrix that is *twice* as ill-conditioned (in digits) as
the design matrix itself. Monomial Vandermonde matrices make this vivid: they grow
ill-conditioned quickly with degree.

1. For Vandermonde design matrices of degree $d = 3, 6, 10$ on
   $x = $ `linspace(0, 1, 30)`, compute $\kappa(A)$ and $\kappa(A^\top A)$
   (`numpy.linalg.cond`).
2. Confirm the second is the square of the first (equivalently,
   $\log_{10}\kappa(A^\top A) = 2\log_{10}\kappa(A)$).
   {numref}`fig-fit-condition` plots both against degree.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    np.log10(cond_AtA),
    2.0 * np.log10(cond_A),
    "forming AᵀA squares the condition number (log₁₀κ(AᵀA) = 2 log₁₀κ(A))",
    atol=0.5,
)

## Exercise 4 — Accuracy loss in practice

The squared condition number is not academic: it
destroys real coefficients. Here we set up a fit with a *known* answer and watch
the normal equations get it wrong while the SVD-based solver gets it right. The
only difference between the two is whether $A^\top A$ is ever formed.

1. Build the degree-10 Vandermonde $V$ on $x = $ `linspace(0, 1, 30)`, choose
   true coefficients $\mathbf c_{\rm true} = $ `default_rng(0).normal(size=11)`,
   and form the exact $\mathbf y = V\mathbf c_{\rm true}$ (no noise — the only
   error will be numerical).
2. Recover $\mathbf c$ two ways: the normal equations
   $V^\top V\mathbf c = V^\top\mathbf y$ and the SVD-based `numpy.linalg.lstsq`.
3. Compare the recovery errors against $\mathbf c_{\rm true}$: the SVD solve
   should be dramatically more accurate.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    err_normal > 100 * err_svd,
    "the SVD-based solve recovers the coefficients far more accurately than the normal equations",
    f"normal {err_normal:.2e}  vs  SVD {err_svd:.2e}",
)

## Exercise 5 — Goodness of fit, χ², and error bars

A fitted number without an uncertainty is half an
answer. From the residuals of a fit we get the RMSE, and from the covariance
{eq}`eq-chi2` we get a **standard error** on every parameter: the $\pm$ that
belongs on every reported slope and intercept.

For the linear fit of Exercise 1 (the same $x$, $y$, and design matrix):

1. Compute the residuals and the RMSE.
2. Form the parameter covariance $\hat\sigma^2 (A^\top A)^{-1}$ with
   $\hat\sigma^2 = \lVert\mathbf r\rVert^2/(n-p)$ ({eq}`eq-chi2`; the small
   $p\times p$ inverse via `numpy.linalg.inv` is legitimate here — it is used for
   uncertainty, not to solve the fit).
3. Report the slope and intercept with their standard errors, and confirm the
   true values $(a, b) = (2, 3)$ lie within a few standard errors of the fit.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.check(
    np.all(within < 3.0),
    "the true parameters lie within three standard errors of the fit",
    f"deviations {within[0]:.2f}σ (a), {within[1]:.2f}σ (b)",
)

## Exercise 6 — Nonlinear least squares

When a parameter sits *inside* a nonlinear function
(a rate inside an exponential), there is no design matrix and no closed form
({eq}`eq-nonlin`). `scipy.optimize.curve_fit` minimises the residual norm
iteratively with Levenberg–Marquardt: each step linearises the model and solves a
least-squares problem, so it is built from everything above, and it is a
root-finding/optimisation iteration in the spirit of [§0.2](root-finding.ipynb)
and [§0.7](ode-solvers.ipynb). It returns the
parameter covariance too, so the error bars come for free.

1. Fit the model $y = A\,e^{-kt} + c$ to the explicit dataset
   $t = $ `linspace(0, 5, 40)`, with true $(A, k, c) = (3, 1.2, 0.5)$ and noise
   `default_rng(0).normal(0, 0.05, 40)`, using `scipy.optimize.curve_fit`.
2. Report the standard errors from the returned covariance.
3. Confirm the recovery is accurate. {numref}`fig-fit-nonlinear` shows the data
   and the fitted curve.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    popt,
    [3.0, 1.2, 0.5],
    "nonlinear least squares recovers the exponential parameters (A, k, c)",
    rtol=0.1,
)

## Exercise 7 — Overfitting (student exercise)

Here is the lesson that never goes away, and it is yours
to build. More parameters always fit the *training* points better, but past a
point they fit the **noise** and generalise worse. The diagnostic is to measure
error on two sets: the data we fit, and a held-out reference we did not. As the
polynomial degree climbs, the training error falls monotonically while the error
against the true curve turns and rises.

Take the explicit data: 12 points $x = $ `linspace(0, 1, 12)`, $y = \sin(2\pi x) +
\varepsilon$ with $\varepsilon = $ `default_rng(1).normal(0, 0.2, 12)`, and a
held-out **test set** on a dense grid `linspace(0, 1, 400)` compared against the
true $\sin(2\pi x)$.

1. Fit polynomials of degree 1, 3, and 9 with `numpy.polynomial.Polynomial.fit`
   (least squares on a rescaled domain, the stable variant).
2. Compute both the training RMSE (against the noisy data) and the test RMSE
   (against the true curve).
3. Show the degree-9 fit drives the training error down while the test error
   rises: it is fitting the noise. {numref}`fig-fit-overfit` shows all three fits
   with the true curve, and the degree-9 wiggle should be unmistakable.

Because the check tests the *errors* (test RMSE rising with degree), a ✗ means
"re-examine the fits or the RMSE computation," never a problem with the plot.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.check(
    test_rmse[9] > test_rmse[3],
    "the highest-degree polynomial generalises worse: it overfits the noise",
    f"test RMSE: degree 9 = {test_rmse[9]:.3f} > degree 3 = {test_rmse[3]:.3f}",
)

## Exercise 8 — Weighted least squares (synthesis)

A closing synthesis: when points carry unequal
uncertainties, an unweighted fit lets the noisy points shout as loudly as the
precise ones. Weighting by $1/\sigma_i^2$ ({eq}`eq-weighted`) is the principled,
maximum-likelihood choice, and in practice it recovers the truth more reliably.

Build explicit heteroscedastic data on the line $2 + 3x$ for
$x = $ `linspace(0, 10, 50)`, with per-point uncertainties $\sigma = $
`linspace(0.5, 2, 50)` and noise drawn at those $\sigma$
(`default_rng(3).normal(0, σ)`).

1. Fit the line **unweighted** (`numpy.linalg.lstsq`).
2. Fit it **weighted**, with $W = \operatorname{diag}(1/\sigma^2)$ implemented as
   a least-squares solve on the row-scaled system (each row of $A$ and $y$
   multiplied by $1/\sigma_i$).
3. Confirm the weighted fit recovers the true parameters $(2, 3)$.

In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.close(
    c_weighted,
    [2.0, 3.0],
    "the weighted fit recovers the true parameters (a, b) = (2, 3)",
    rtol=0.3,
)

## Notebook summary

- Linear least squares three ways (normal equations, QR, SVD) and the design-matrix projection
  picture; why the normal equations square the condition number, and the accuracy loss that
  follows.
- Goodness of fit with $\chi^2$ and error bars; nonlinear least squares
  (`scipy.optimize.curve_fit`); overfitting; and weighted least squares.

## Outlook

- **Regularisation (ridge / Tikhonov).** When a fit is ill-posed, adding a penalty
  $\lambda\lVert\mathbf c\rVert^2$ stabilises it, and connects directly to the
  truncated-SVD low-rank idea of [§0.5](eigenvalues-svd.ipynb).
- **Total least squares.** When there is error in $x$ as well as $y$, ordinary
  least squares is biased; orthogonal-distance regression fits the perpendicular
  residuals instead.
- **Better bases.** The monomial Vandermonde of Exercise 3 is badly conditioned, and
  there are two distinct cures. Orthogonal polynomials fix it outright
  (`numpy.polynomial.Chebyshev.fit`), while `numpy.polynomial.Polynomial.fit` softens
  it a different way, rescaling the data window to $[-1, 1]$ in the monomial basis.
- **What the error bars mean.** Bayesian curve fitting reinterprets the covariance
  as a posterior: a forward link to the statistical-mechanics volume.
- **Everywhere downstream.** Fitting extracts rates, exponents, lattice constants,
  and spectroscopic peaks; this notebook is the machine under all of it.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()